In [1]:
# ============================================================================
# BLOCK 6: UMAP VISUALIZATION - ROMANIAN
# Input: 05_topics_data_ro.pkl
# Output: 06_umap_data_ro.pkl + PNG visualizations
# Runtime: ~10-15 minutes on CPU
# ============================================================================
%run 00_setup_and_config.ipynb

c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\election_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 11:47:41
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Romanian stopwords: 385 words
✓ Polish stopwords: 511 words
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_ana

In [2]:
print('='*70)
print('BLOCK 6: UMAP VISUALIZATION - ROMANIAN')
print('='*70)

if check_checkpoint_exists('ro', '06_umap_data'):
    df_ro = load_checkpoint('ro', '06_umap_data')
    print('✓ Loading from UMAP checkpoint')
else:
    df_ro = load_checkpoint('ro', '05_topics_data')
    if df_ro is None:
        raise FileNotFoundError('Run 04_topic_modeling_ro.ipynb first')

print(f'\nComputing UMAP projection for {len(df_ro):,} comments...')

# Embeddings
print('\nComputing sentence embeddings...')

embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embedding_model = embedding_model.to('cpu')

texts = df_ro['clean_text'].fillna('').astype(str).tolist()

embeddings = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Embedding'):
    batch_texts = texts[i:i+BATCH_SIZE]
    batch_embeddings = embedding_model.encode(batch_texts, convert_to_numpy=True)
    embeddings.append(batch_embeddings)

embeddings = np.vstack(embeddings)
print(f'✓ Embeddings shape: {embeddings.shape}')

# UMAP
print('\nRunning UMAP dimensionality reduction...')

reducer = umap.UMAP(
    n_neighbors=UMAP_CONFIG['n_neighbors'],
    min_dist=UMAP_CONFIG['min_dist'],
    n_components=UMAP_CONFIG['n_components'],
    metric=UMAP_CONFIG['metric'],
    random_state=UMAP_CONFIG['random_state'],
    verbose=True
)

umap_embeddings = reducer.fit_transform(embeddings)

df_ro['umap_x'] = umap_embeddings[:, 0]
df_ro['umap_y'] = umap_embeddings[:, 1]

save_checkpoint(df_ro, 'ro', '06_umap_data')
update_pipeline_status('ro', 6, 'completed')

# Visualization
print('\nGenerating UMAP visualizations...')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'UMAP Projection - Romanian Comments (n={len(df_ro):,})', fontsize=14, fontweight='bold')

scatter1 = axes[0].scatter(df_ro['umap_x'], df_ro['umap_y'], 
                           c=df_ro['topic'], cmap='Set3', alpha=0.6, s=10)
axes[0].set_xlabel('UMAP Dimension 1')
axes[0].set_ylabel('UMAP Dimension 2')
axes[0].set_title('Colored by Topic')
axes[0].grid(alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='Topic')

sentiment_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
df_ro['sentiment_numeric'] = df_ro['sentiment'].map(sentiment_map).fillna(1)
scatter2 = axes[1].scatter(df_ro['umap_x'], df_ro['umap_y'], 
                           c=df_ro['sentiment_numeric'], 
                           cmap='RdYlGn', alpha=0.6, s=10)
axes[1].set_xlabel('UMAP Dimension 1')
axes[1].set_ylabel('UMAP Dimension 2')
axes[1].set_title('Colored by Sentiment')
axes[1].grid(alpha=0.3)
plt.colorbar(scatter2, ax=axes[1], label='Sentiment')

plt.tight_layout()
save_figure(fig, 'ro_umap_scatter.png', 'ro', 'visualizations')

print('\n' + '='*70)
print('✓ BLOCK 6 COMPLETE - UMAP VISUALIZATION DONE')
print('='*70)


BLOCK 6: UMAP VISUALIZATION - ROMANIAN
✓ Loading checkpoint: 05_topics_data_ro.pkl

Computing UMAP projection for 6,770 comments...

Computing sentence embeddings...


Embedding: 100%|██████████| 847/847 [05:05<00:00,  2.78it/s]


✓ Embeddings shape: (6770, 384)

Running UMAP dimensionality reduction...
UMAP(angular_rp_forest=True, metric='cosine', n_jobs=1, random_state=42, verbose=True)
Fri May 29 11:52:54 2026 Construct fuzzy simplicial set
Fri May 29 11:52:54 2026 Finding Nearest Neighbors
Fri May 29 11:52:54 2026 Building RP forest with 9 trees
Fri May 29 11:53:01 2026 NN descent for 13 iterations
	 1  /  13
	 2  /  13
	 3  /  13
	 4  /  13
	 5  /  13
	Stopping threshold met -- exiting after 5 iterations
Fri May 29 11:53:16 2026 Finished Nearest Neighbor Search
Fri May 29 11:53:21 2026 Construct embedding


Epochs completed:   2%| ▏          11/500 [00:01]

	completed  0  /  500 epochs


...

	completed  400  /  500 epochs


Epochs completed:  92%| █████████▏ 458/500 [00:12]

	completed  450  /  500 epochs


Epochs completed: 100%| ██████████ 500/500 [00:13]


Fri May 29 11:53:35 2026 Finished embedding
✓ Checkpoint saved: 06_umap_data_ro.pkl
✓ Pipeline status updated: ro - Block 6 - completed

Generating UMAP visualizations...
✓ Saved: outputs\ro\visualizations\ro_umap_scatter.png

✓ BLOCK 6 COMPLETE - UMAP VISUALIZATION DONE
